# Formation Energy of the Mg_Ga-V_N Defect Pair in GaN

> **G. Miceli, A. Pasquarello**
> "Self-compensation due to point defects in Mg-doped GaN"
> Phys. Rev. B 93, 165207 (2016)
> [DOI: 10.1103/PhysRevB.93.165207](https://doi.org/10.1103/PhysRevB.93.165207)

Computes the neutral formation energy of the axial Mg_Ga-V_N pair created in the
[structure notebook](defect_point_pair_gallium_nitride.ipynb), with Miceli & Pasquarello's chemical
potentials (Sec. II A): mu_Ga + mu_N = mu_GaN in both limits and mu_Mg = (mu_Mg3N2 - 2 mu_N) / 3, so
E_f = E(pair) - E(pristine) + mu_GaN - mu_Mg comes out at the Ga-rich and the N-rich limit. Those two
values and the formation enthalpy of GaN are compared with Miceli's Table I.

<h2 style="color:green">Usage</h2>

1. Create the materials in the [structure notebook](defect_point_pair_gallium_nitride.ipynb), which saves `GaN 3x3x2` and `GaN 3x3x2 Mg_Ga-V_N axial pair` to the `uploads` folder.
1. Set the material names and parameters in cells 1.2-1.4 (or use the defaults).
1. Click "Run" > "Run All" to run all cells.
1. Wait for the jobs to complete.
1. Scroll down to view the results.

## Summary

1. Set up the environment and parameters: install packages (JupyterLite only) and configure the material names, model and compute parameters.
1. Authenticate and initialize API client: authenticate via browser, initialize the client, then select account and project.
1. Load the pair material by name, load the GaN, Ga, N2 and Mg3N2 references from Standata, print provenance and the Mg-N bond lengths, then save all five to the platform.
1. Configure the shared DFT model and k-grid: one model and a per-material k-grid for every workflow below.
1. Configure compute: get the list of clusters and create a compute configuration.
1. Create the missing fixed-cell relaxation jobs for the four references and wait for them.
1. Create, submit and monitor the fixed-cell relaxation job on the pair cell.
1. Retrieve the results: the relaxed total energies, the chemical potentials in both limits, the formation enthalpy of GaN and the formation energy of the pair.
1. Compare with Miceli & Pasquarello (2016).

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|specific_examples|api_examples")


### 1.2. Material names

In [ ]:
# Name saved by defect_point_pair_gallium_nitride.ipynb.
DEFECTIVE_NAME = "GaN 3x3x2 Mg_Ga-V_N axial pair"
GAN_REFERENCE_NAME = "GaN"                                          # Standata, mp-804 unit cell, mu_GaN
GA_REFERENCE_NAME = "Ga-[Gallium]-ORC_[Cmce]_3D_[Bulk]-[mp-142]"    # Standata, mu_Ga at the Ga-rich limit
N2_REFERENCE_NAME = "N2-[Nitrogen]-FCC_[P2_13]_3D_[Bulk]-[mp-154]"  # Standata, mu_N at the N-rich limit
# paper: 3 mu_Mg + 2 mu_N = mu_Mg3N2 (Sec. II A), pinning mu_Mg
MG3N2_REFERENCE_NAME = "Mg3N2-[Magnesium_Nitride]-BCC_[Ia-3]_3D_[Bulk]-[mp-1559]"


### 1.3. Parameters

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

ORGANIZATION_NAME = None  # set to your organization name (full or partial); otherwise, your default one is used
FOLDER = "./uploads"

RELAX_WORKFLOW_SEARCH_TERM = "fixed_cell_relaxation.json"
APPLICATION_NAME = "espresso"

CLUSTER_NAME = None
QUEUE_NAME = QueueName.OF
PPN = 40
TIME_LIMIT = "12:00:00"  # the seminar account refuses to pre-authorize 24 h x 40 ppn; jobs are restartable, so a long relaxation continues past this
TOLERANCE_FRACTION = 0.15  # fractional agreement with the paper that counts as reproduced

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 60  # seconds


### 1.4. DFT model parameters

In [ ]:
FUNCTIONAL = "hse06"
MODEL_SUBTYPE = "hybrid"
# paper: norm-conserving, Ga 3d in core, 45 Ry. Ours: GBRV ultrasoft with Ga 3d in valence at its own
# converged 40/200 Ry - the paper's 45 Ry belongs to its pseudopotentials and does not transfer
# (PseudoDojo NC at 45 Ry gave 0.7 eV/A on the perfect crystal). To match the type, use "nc" at that
# set's converged cutoff.
PSEUDOPOTENTIAL_TYPE = "us"
ECUTWFC = 40   # Ry
ECUTRHO = 200  # Ry

KPOINT_DENSITY = 4  # Å⁻¹, for the Ga and N2 references; the grid it gives is printed per material
# paper: Gamma for the relaxation, 2x2x2 MP for the final energies; ours Gamma throughout for speed.
# To match, set the pair k-grid and nqx to [2, 2, 2] for the energy step.
GAMMA_KGRID = [1, 1, 1]
# paper: 96-atom supercell (matrix not given); ours 3x3x2 = 72 atoms for speed. To match, build a
# 96-atom cell in the structure notebook and fold its k-grid here.
SUPERCELL_KGRID = [3, 3, 2]   # the pair supercell's Gamma point folded back onto the GaN unit cell
SUPERCELL_FACTOR = 3 * 3 * 2  # unit cells per supercell: E(pristine) = 18 x E(GaN unit cell)
MODEL_TAG = f"{FUNCTIONAL}-{PSEUDOPOTENTIAL_TYPE} {ECUTWFC}-{ECUTRHO}Ry k{KPOINT_DENSITY}"

RELAX_UNIT = "pw_relax"
HSE_SETTINGS = {"input_dft": "hse", "exx_fraction": 0.31}  # the paper's HSE with 31% Fock exchange
# paper: spin-unrestricted where unpaired electrons occur; the neutral pair is closed-shell
# (PBE: 0.00 muB), so nspin = 1. To check at HSE, patch {"nspin": 2, "starting_magnetization(2)": 0.5}
# (species 2 = N) into the pair's &SYSTEM.
# paper: fully relaxed, threshold not stated; 0.05 eV/A is ours.
RELAXATION_SETTINGS = {"forc_conv_thr": 1.9e-3, "nstep": 100}
# Names the relaxation jobs.
RELAX_TAG = f"{MODEL_TAG} f{RELAXATION_SETTINGS['forc_conv_thr']}"


## 2. Authenticate and initialize API client
### 2.1. Authenticate

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()


### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client


### 2.3. Select account

In [ ]:
client.list_accounts()


In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")


### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")


## 3. Load the materials
### 3.1. Load the pair from the uploads folder or the platform, load the references from Standata, and print provenance

In [ ]:
from collections import Counter
from mat3ra.made.material import Material
from mat3ra.notebooks_utils.core.entity.material.api import load_material
from mat3ra.standata.materials import Materials

pair = load_material(client, FOLDER, DEFECTIVE_NAME, ACCOUNT_ID)
gan_reference = Material.create(Materials.get_by_name_first_match(GAN_REFERENCE_NAME))
ga_reference = Material.create(Materials.get_by_name_first_match(GA_REFERENCE_NAME))
n2_reference = Material.create(Materials.get_by_name_first_match(N2_REFERENCE_NAME))
mg3n2_reference = Material.create(Materials.get_by_name_first_match(MG3N2_REFERENCE_NAME))
materials_by_name = {DEFECTIVE_NAME: pair, GAN_REFERENCE_NAME: gan_reference,
                     GA_REFERENCE_NAME: ga_reference, N2_REFERENCE_NAME: n2_reference,
                     MG3N2_REFERENCE_NAME: mg3n2_reference}

for name, material in materials_by_name.items():
    a, c = material.lattice.a, material.lattice.c
    composition = "".join(f"{e}{n}" for e, n in sorted(Counter(material.basis.elements.values).items()))
    print(f"{name}: {composition}, {material.basis.number_of_atoms} atoms, cell {a:.2f} x {c:.2f} Å")


### 3.2. Mg-N bond lengths

Mg substitutes a Ga site, which carries four N neighbours in wurtzite; the axial one is the vacancy,
so the pair cell keeps three Mg-N bonds and the fourth N sits a full lattice vector away (FIG. 2(c)).

In [ ]:
import math

pair_cartesian = pair.clone()
pair_cartesian.to_cartesian()
sites = list(zip(pair_cartesian.basis.coordinates.values, pair_cartesian.basis.elements.values))
(magnesium_site,) = [tuple(site) for site, element in sites if element == "Mg"]
nitrogen_distances = sorted(math.dist(magnesium_site, tuple(site)) for site, element in sites if element == "N")
print("Mg-N distances: " + ", ".join(f"{distance:.3f}" for distance in nitrogen_distances[:4]) + " Å")


### 3.3. Save the materials to the platform

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

saved_pair = get_or_create_material(client, pair, ACCOUNT_ID)
saved_gan_reference = get_or_create_material(client, gan_reference, ACCOUNT_ID)
saved_ga_reference = get_or_create_material(client, ga_reference, ACCOUNT_ID)
saved_n2_reference = get_or_create_material(client, n2_reference, ACCOUNT_ID)
saved_mg3n2_reference = get_or_create_material(client, mg3n2_reference, ACCOUNT_ID)


## 4. Configure the shared DFT model and k-grid
### 4.1. DFT model

In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.standata.model_tree import ModelTreeStandata
from mat3ra.ade.application import Application
from mat3ra.mode import ModelFactory

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)

model_config = ModelTreeStandata.get_model_by_parameters(type="dft", subtype=MODEL_SUBTYPE, functional=FUNCTIONAL)
model_config["method"] = {"type": "pseudopotential", "subtype": PSEUDOPOTENTIAL_TYPE}
model = ModelFactory.create(model_config)
print(f"Using application: {app.name}, model: {MODEL_TAG}")


### 4.2. k-grid per material

The pair supercell is sampled at Γ; the pristine reference is the GaN unit cell on the Γ-centred
3x3x2 grid those Γ points fold back to, so E(pristine) = 18 x E(unit cell) carries the same sampling
error as the pair and it cancels in E(pair) - E(pristine).

In [ ]:
from mat3ra.notebooks_utils.workflow import kgrid_from_density

material_objects = {DEFECTIVE_NAME: pair, GAN_REFERENCE_NAME: gan_reference,
                    GA_REFERENCE_NAME: ga_reference, N2_REFERENCE_NAME: n2_reference,
                    MG3N2_REFERENCE_NAME: mg3n2_reference}

kgrid = {name: kgrid_from_density(material, KPOINT_DENSITY) for name, material in material_objects.items()}
kgrid[DEFECTIVE_NAME] = GAMMA_KGRID
kgrid[GAN_REFERENCE_NAME] = SUPERCELL_KGRID
exx_settings = {name: {**HSE_SETTINGS, "nqx1": grid[0], "nqx2": grid[1], "nqx3": grid[2]}
                for name, grid in kgrid.items()}
for name, grid in kgrid.items():
    print(f"{name}: k-grid {grid}")


## 5. Create the compute configuration
### 5.1. Select cluster

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")


### 5.2. Create the compute configuration for the jobs

In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
    if cluster is None:
        raise ValueError(f"Cluster '{CLUSTER_NAME}' not found. Available: {[c['hostname'] for c in clusters]}")
else:
    cluster = clusters[0]
compute = Compute(cluster=cluster, queue=QUEUE_NAME, ppn=PPN, timeLimit=TIME_LIMIT)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}, "
      f"time limit: {TIME_LIMIT}")


## 6. Fixed-cell relaxation jobs for the reference materials

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.notebooks_utils.workflow import apply_planewave_cutoffs, apply_scf_kgrid, patch_workflow_qe_input
from mat3ra.notebooks_utils.job import create_job
from mat3ra.notebooks_utils.core.entity.job.api import find_job_for_material

relaxation_workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(
    RELAX_WORKFLOW_SEARCH_TERM
)
prerequisite_materials = {GAN_REFERENCE_NAME: saved_gan_reference, GA_REFERENCE_NAME: saved_ga_reference,
                          N2_REFERENCE_NAME: saved_n2_reference,
                          MG3N2_REFERENCE_NAME: saved_mg3n2_reference}
prerequisite_job_ids = {}
new_job_ids = []
for name, saved_material in prerequisite_materials.items():
    workflow = Workflow.create(relaxation_workflow_config)
    workflow.name = f"Fixed-cell Relaxation {name} {RELAX_TAG}"
    workflow.subworkflows[0].model = model
    apply_planewave_cutoffs(workflow, ECUTWFC, ECUTRHO, unit_name=RELAX_UNIT)
    apply_scf_kgrid(workflow, kgrid[name], material=material_objects[name], unit_name=RELAX_UNIT)
    patch_workflow_qe_input(workflow, {"system": exx_settings[name]}, [RELAX_UNIT])
    patch_workflow_qe_input(workflow, {"control": RELAXATION_SETTINGS}, [RELAX_UNIT])
    job = find_job_for_material(client, saved_material["_id"], workflow.name, ACCOUNT_ID,
                                statuses=("submitted", "queued", "active", "finished"))
    if job is None:
        job = create_job(
            api_client=client, materials=[saved_material], workflow=workflow, project_id=project_id,
            owner_id=ACCOUNT_ID, compute=compute.to_dict(), prefix=f"{workflow.name} {timestamp}",
        )
        new_job_ids.append(job["_id"])
        print(f"✅ {name}: created relaxation job {job['_id']}")
    else:
        print(f"♻️  {name}: reusing existing relaxation job {job['_id']}")
    prerequisite_job_ids[name] = job["_id"]


In [ ]:
from mat3ra.notebooks_utils.api.job import submit_jobs, wait_for_jobs_to_finish_async

if new_job_ids:
    submit_jobs(client.jobs, new_job_ids)
    print(f"✅ Submitted {len(new_job_ids)} prerequisite job(s).")
    await wait_for_jobs_to_finish_async(client.jobs, new_job_ids, poll_interval=POLL_INTERVAL)


## 7. Fixed-cell relaxation of the pair cell
### 7.1. Configure the workflow

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow

pair_workflow = Workflow.create(relaxation_workflow_config)
pair_workflow.name = f"Fixed-cell Relaxation {DEFECTIVE_NAME} {RELAX_TAG}"
pair_workflow.subworkflows[0].model = model
apply_planewave_cutoffs(pair_workflow, ECUTWFC, ECUTRHO, unit_name=RELAX_UNIT)
apply_scf_kgrid(pair_workflow, kgrid[DEFECTIVE_NAME], material=pair, unit_name=RELAX_UNIT)
patch_workflow_qe_input(pair_workflow, {"system": exx_settings[DEFECTIVE_NAME]}, [RELAX_UNIT])
patch_workflow_qe_input(pair_workflow, {"control": RELAXATION_SETTINGS}, [RELAX_UNIT])

visualize_workflow(pair_workflow)


### 7.2. Create, submit and monitor the job

In [ ]:
pair_job = find_job_for_material(
    client, saved_pair["_id"], pair_workflow.name, ACCOUNT_ID, statuses=("submitted", "queued", "active", "finished")
)
if pair_job is None:
    pair_job = create_job(
        api_client=client, materials=[saved_pair], workflow=pair_workflow,
        project_id=project_id, owner_id=ACCOUNT_ID, compute=compute.to_dict(),
        prefix=f"{pair_workflow.name} {timestamp}",
    )
    submit_jobs(client.jobs, [pair_job["_id"]])
    print(f"✅ Relaxation job created and submitted: {pair_job['_id']}")
else:
    print(f"♻️  Reusing existing relaxation job: {pair_job['_id']}")
pair_job_id = pair_job["_id"]
await wait_for_jobs_to_finish_async(client.jobs, [pair_job_id], poll_interval=POLL_INTERVAL)


## 8. Retrieve the results
### 8.1. Relaxed total energies and chemical potentials

In [ ]:
from mat3ra.notebooks_utils.core.entity.property.api import get_properties_for_job

e_gan_unit = get_properties_for_job(client, prerequisite_job_ids[GAN_REFERENCE_NAME], "total_energy")[0]["value"]
e_ga = get_properties_for_job(client, prerequisite_job_ids[GA_REFERENCE_NAME], "total_energy")[0]["value"]
e_n2 = get_properties_for_job(client, prerequisite_job_ids[N2_REFERENCE_NAME], "total_energy")[0]["value"]
e_mg3n2 = get_properties_for_job(client, prerequisite_job_ids[MG3N2_REFERENCE_NAME], "total_energy")[0]["value"]
e_pair = get_properties_for_job(client, pair_job_id, "total_energy")[0]["value"]
total_force = get_properties_for_job(client, pair_job_id, "total_force")[0]

e_pristine = SUPERCELL_FACTOR * e_gan_unit
mu_gan = e_gan_unit / Counter(gan_reference.basis.elements.values)["Ga"]
mu_mg3n2 = e_mg3n2 / (Counter(mg3n2_reference.basis.elements.values)["Mg"] / 3)
mu_ga_ga_rich = e_ga / ga_reference.basis.number_of_atoms
mu_n_n_rich = e_n2 / n2_reference.basis.number_of_atoms
mu_n_ga_rich = mu_gan - mu_ga_ga_rich
mu_ga_n_rich = mu_gan - mu_n_n_rich
mu_mg_ga_rich = (mu_mg3n2 - 2 * mu_n_ga_rich) / 3
mu_mg_n_rich = (mu_mg3n2 - 2 * mu_n_n_rich) / 3

print(f"Residual force on the relaxed pair: {total_force['value']:.4f} {total_force['units']}")
print(f"μ_GaN: {mu_gan:.4f} eV/f.u., μ_Mg3N2: {mu_mg3n2:.4f} eV/f.u.")
print(f"Ga-rich: μ_Ga = {mu_ga_ga_rich:.4f}, μ_N = {mu_n_ga_rich:.4f}, μ_Mg = {mu_mg_ga_rich:.4f} eV/atom")
print(f"N-rich:  μ_Ga = {mu_ga_n_rich:.4f}, μ_N = {mu_n_n_rich:.4f}, μ_Mg = {mu_mg_n_rich:.4f} eV/atom")


### 8.2. Formation enthalpy of GaN and formation energy of the pair

mu_Mg3N2 pins mu_Mg at each limit, and it cancels out of E_f(N-rich) - E_f(Ga-rich) =
(2/3) (mu_N(N-rich) - mu_N(Ga-rich)), which is therefore fixed by the formation enthalpy of GaN alone.

In [ ]:
formation_enthalpy_gan = mu_gan - mu_ga_ga_rich - mu_n_n_rich
e_formation_splitting = (2 / 3) * (mu_n_n_rich - mu_n_ga_rich)
e_formation_ga_rich = e_pair - e_pristine + mu_gan - mu_mg_ga_rich
e_formation_n_rich = e_pair - e_pristine + mu_gan - mu_mg_n_rich

print(f"ΔH_f(GaN): {formation_enthalpy_gan:.3f} eV per formula unit")
print(f"E_f(Ga-rich): {e_formation_ga_rich:.3f} eV, E_f(N-rich): {e_formation_n_rich:.3f} eV")
print(f"E_f(N-rich) - E_f(Ga-rich): {e_formation_splitting:.3f} eV")


## 9. Comparison with Miceli & Pasquarello (2016)

The model is the paper's HSE with 31% Fock exchange; the basis is GBRV ultrasoft at 40/200 Ry with
gallium 3d in valence, and every cell below is relaxed at fixed shape and volume.

In [ ]:
# eV. Table I labels both complex rows E_f^{+2}; Fig. 3 identifies the flat 1.2 / 2.1 eV values above
# the +2/0 transition at E_F = 0.80 eV as the neutral ones. ΔH_f(GaN) is Table I's own V_N rows,
# 4.7 (N-rich) - 3.3 (Ga-rich), since V_N carries ΔN_N = -1.
MICELI = {"e_f_ga_rich": 1.2, "e_f_n_rich": 2.1, "formation_enthalpy_gan": 1.4}

relative_ga_rich = abs(e_formation_ga_rich - MICELI["e_f_ga_rich"]) / MICELI["e_f_ga_rich"]
relative_n_rich = abs(e_formation_n_rich - MICELI["e_f_n_rich"]) / MICELI["e_f_n_rich"]
verdict = "yes" if (max(relative_ga_rich, relative_n_rich) <= TOLERANCE_FRACTION
                    and e_formation_ga_rich < e_formation_n_rich) else "no"
print(f"E_f(Ga-rich): {e_formation_ga_rich:7.3f} eV, Miceli {MICELI['e_f_ga_rich']:.3f} eV ({relative_ga_rich:.0%})")
print(f"E_f(N-rich):  {e_formation_n_rich:7.3f} eV, Miceli {MICELI['e_f_n_rich']:.3f} eV ({relative_n_rich:.0%})")
print(f"|ΔH_f(GaN)|:  {abs(formation_enthalpy_gan):7.3f} eV, Miceli {MICELI['formation_enthalpy_gan']:.3f} eV "
      f"(secondary check)")
print(f"E_f(N-rich) - E_f(Ga-rich): {e_formation_splitting:.3f} eV, "
      f"Miceli {MICELI['e_f_n_rich'] - MICELI['e_f_ga_rich']:.3f} eV")
print(f"Reproduces Miceli & Pasquarello (2016): {verdict} (E_f Ga-rich/N-rich, relaxed HSE)")


## References

[1] Miceli, G., & Pasquarello, A. (2016). Self-compensation due to point defects in Mg-doped GaN. Phys. Rev. B, 93(16), 165207. https://doi.org/10.1103/PhysRevB.93.165207